# Raport z Listy 2: Sztuczna Inteligencja i Inżynieria Wiedzy

**Temat:** Implementacja algorytmów Minimax i Alfa-Beta dla gry Clobber

**Autor:** Mikołaj Kubś 272662

**Data:** 5.05.2025

## Clobber

Clobber to abstrakcyjna gra planszowa dla dwóch graczy, rozgrywana na siatce (np. 8×8). Na początku każdy gracz ma swoje pionki umieszczone na polach w ich kolorze na szachownicy. Gracze na zmianę wykonują ruchy: przesuwają jeden ze swoich pionków na pole sąsiadujące w pionie lub poziomie, zajmując miejsce pionka przeciwnika — "clobberując" go, czyli usuwając z planszy.

Celem gry jest unieruchomienie przeciwnika — gracz przegrywa, jeśli nie może wykonać legalnego ruchu.

## Opis Teoretyczny Metody

### Teoria Gier i Gry o Sumie Zerowej

**Teoria gier** jest dziedziną matematyki i ekonomii zajmującą się analizą strategicznych interakcji między racjonalnymi decydentami (graczami). Formalnie, grę można zdefiniować jako trójkę `G = (N, S, U)`, gdzie (zgodnie z Definicją 2 z treści zadania):
*   `N` oznacza liczbę graczy. W przypadku Clobber `N = 2`.
*   `S = S1 × S2 × ... × SN` jest zbiorem możliwych profili strategii, gdzie `Si` to zbiór strategii (lub w kontekście drzewa gry, sekwencji akcji) dostępnych dla gracza `i`. W praktyce, w grach takich jak Clobber, częściej myślimy o przestrzeni stanów gry i możliwych przejściach (ruchach) między nimi.
*   `U = (u1, u2, ..., uN)` jest zbiorem funkcji użyteczności (wypłat), gdzie `ui: S → R` przypisuje wartość liczbową (wypłatę) graczowi `i` dla każdego możliwego wyniku gry (osiągniętego przez kombinację strategii `s ∈ S`).

**Gra o sumie zerowej** to szczególny typ gry, w której suma wypłat wszystkich graczy dla dowolnego wyniku gry jest równa zero:
`∀s ∈ S, Σ(i∈N) ui(s) = 0`

Oznacza to, że zysk jednego gracza jest dokładnie równy stracie drugiego (lub sumie strat pozostałych graczy, jeśli N > 2). W grach dwuosobowych, jak Clobber, jeśli zdefiniujemy wypłatę za wygraną jako +1, za przegraną jako -1, a za remis jako 0, to warunek sumy zerowej jest spełniony (1 + (-1) = 0). Gry o sumie zerowej charakteryzują się czystym konfliktem interesów między graczami. Algorytmy takie jak Minimax są szczególnie dobrze dopasowane do tego typu gier.

### Drzewo Decyzyjne

W kontekście gier dwuosobowych, **drzewo decyzyjne** (lub drzewo gry) jest fundamentalną strukturą używaną do modelowania wszystkich możliwych sekwencji ruchów. Zgodnie z Definicją 1:
*   Jest to **skierowane drzewo**.
*   Każdy **węzeł** reprezentuje unikalny **stan gry** (np. układ pionków na planszy Clobber). Korzeń drzewa odpowiada początkowemu stanowi gry.
*   Każda **krawędź** reprezentuje **możliwy ruch** (akcję), który prowadzi z jednego stanu do drugiego. Krawędzie wychodzące z danego węzła odpowiadają wszystkim legalnym ruchom, jakie gracz, którego jest tura w tym stanie, może wykonać.
*   **Liście** drzewa reprezentują **stany końcowe gry** (wygrana jednego z graczy lub remis) lub stany osiągnięte na **maksymalnej głębokości przeszukiwania**, jeśli pełne drzewo jest zbyt duże do analizy.

**Charakterystyka drzewa gry:**
*   **Rozgałęzienie (Branching Factor):** Liczba krawędzi wychodzących z węzła (stopień wyjściowy) odpowiada liczbie możliwych ruchów w danym stanie. Może ona być różna dla różnych węzłów, nawet na tym samym poziomie.
*   **Głębokość:** Długość ścieżki od korzenia do węzła odpowiada liczbie wykonanych ruchów (pół-rund). Maksymalna głębokość drzewa może być różna dla różnych gałęzi.
*   **Poziomy naprzemienne:** W grach turowych, jak Clobber, poziomy drzewa często odpowiadają naprzemiennie ruchom jednego i drugiego gracza (np. poziom 0 - ruch gracza MAX, poziom 1 - ruch gracza MIN, poziom 2 - ruch gracza MAX, itd.).

### Algorytm Minimax

Algorytm **Minimax** jest podstawowym algorytmem przeszukiwania drzewa gry, stosowanym do podejmowania optymalnych decyzji w dwuosobowych grach o sumie zerowej z pełną informacją (jak szachy, warcaby, Go czy Clobber). Jego celem jest znalezienie ruchu, który maksymalizuje potencjalną wygraną gracza (nazywanego MAX), zakładając, że przeciwnik (nazywany MIN) będzie zawsze wybierał ruchy minimalizujące wygraną gracza MAX (czyli maksymalizujące własną wygraną).

**Zasada działania:**
1.  **Generowanie Drzewa:** Algorytm eksploruje drzewo gry, zaczynając od bieżącego stanu (korzenia). Zwykle przeszukiwanie odbywa się metodą w głąb (DFS).
2.  **Ograniczenie Głębokości:** Ze względu na potencjalnie ogromny rozmiar drzewa gry, przeszukiwanie jest często ograniczone do pewnej maksymalnej głębokości `d`.
3.  **Funkcja Oceny (Heurystyka):**
    *   Dla **liści** drzewa reprezentujących **koniec gry**, wartością jest rzeczywisty wynik gry (np. +1 dla wygranej MAX, -1 dla wygranej MIN, 0 dla remisu).
    *   Dla węzłów osiągniętych na **limicie głębokości `d`**, stosuje się **funkcję heurystyczną**, która szacuje "wartość" danego stanu gry z perspektywy gracza MAX. Dobra heurystyka powinna korelować z szansami na wygraną.
4.  **Propagacja Wartości (Rekurencyjne Obliczenia):** Wartości są obliczane od liści w górę do korzenia:
    *   W węzłach należących do **gracza MAX**, wybierana jest maksymalna wartość spośród wartości jego potomków. `Wartość(MAX_węzeł) = max(Wartość(potomek1), Wartość(potomek2), ...)`
    *   W węzłach należących do **gracza MIN**, wybierana jest minimalna wartość spośród wartości jego potomków. `Wartość(MIN_węzeł) = min(Wartość(potomek1), Wartość(potomek2), ...)`
5.  **Wybór Ruchu:** Po obliczeniu wartości dla wszystkich potomków korzenia, gracz MAX wybiera ruch prowadzący do potomka o najwyższej obliczonej wartości Minimax.

Algorytm Minimax gwarantuje znalezienie optymalnego ruchu przy założeniu optymalnej gry przeciwnika, o ile przeszukane zostanie całe drzewo lub użyta zostanie idealna funkcja heurystyczna. Jego główną wadą jest złożoność czasowa, która wynosi `O(b^d)`, gdzie `b` to średni współczynnik rozgałęzienia, a `d` to głębokość przeszukiwania.


```py
def minimax(node, depth, maximizingPlayer):
    if depth == 0 or node is a terminal node:
        return evaluate(node)
    
    if maximizingPlayer:
        maxEval = -infinity
        for child in node.children():
            eval = minimax(child, depth - 1, False)
            maxEval = max(maxEval, eval)
        return maxEval
    else:
        minEval = +infinity
        for child in node.children():
            eval = minimax(child, depth - 1, True)
            minEval = min(minEval, eval)
        return minEval
```

### Alfa-Beta Cięcie

**Alfa-Beta cięcie (Alpha-Beta Pruning)** jest optymalizacją algorytmu Minimax, która pozwala na znaczne zredukowanie liczby węzłów do przejrzenia w drzewie gry, nie zmieniając przy tym ostatecznego wyniku (wybranego ruchu). Działa na zasadzie "odcinania" całych gałęzi drzewa, co do których można udowodnić, że nie wpłyną one na ostateczną decyzję.

**Kluczowe Elementy:**
*   **Wartości Alfa (α):** Reprezentuje **najlepszą (najwyższą) wartość**, jaką gracz **MAX** może sobie zagwarantować na ścieżce od korzenia do aktualnego węzła. Początkowo `α = -∞`. Wartość `α` może być tylko zwiększana.
*   **Wartości Beta (β):** Reprezentuje **najlepszą (najniższą) wartość**, jaką gracz **MIN** może sobie zagwarantować na ścieżce od korzenia do aktualnego węzła. Początkowo `β = +∞`. Wartość `β` może być tylko zmniejszana.
*   **Przekazywanie Wartości:** Wartości `α` i `β` są przekazywane w dół drzewa podczas rekurencyjnego przeszukiwania.

**Warunki Cięcia:**
1.  **Cięcie Beta (w węźle MAX):** Podczas eksploracji potomków węzła MAX, jeśli obliczona wartość `v` dla któregoś potomka jest **większa lub równa `β` (`v ≥ β`)**, oznacza to, że gracz MIN (na wyższym poziomie) ma już dostępną inną ścieżkę gwarantującą mu wynik co najwyżej `β`. MIN nigdy nie pozwoliłby graczowi MAX osiągnąć wartości `v` (która jest ≥ `β`), więc dalsze przeszukiwanie pozostałych potomków tego węzła MAX jest zbędne. Przeszukiwanie tej gałęzi jest przerywane, a węzeł MAX zwraca co najmniej `β`.
2.  **Cięcie Alfa (w węźle MIN):** Podczas eksploracji potomków węzła MIN, jeśli obliczona wartość `v` dla któregoś potomka jest **mniejsza lub równa `α` (`v ≤ α`)**, oznacza to, że gracz MAX (na wyższym poziomie) ma już dostępną inną ścieżkę gwarantującą mu wynik co najmniej `α`. MAX nigdy nie wybrałby ścieżki prowadzącej do wartości `v` (która jest ≤ `α`), więc dalsze przeszukiwanie pozostałych potomków tego węzła MIN jest zbędne. Przeszukiwanie tej gałęzi jest przerywane, a węzeł MIN zwraca co najwyżej `α`.

**Korzyści:**
Alfa-Beta cięcie w optymalnym przypadku (przy idealnym uporządkowaniu ruchów – najlepsze ruchy sprawdzane jako pierwsze) może zredukować efektywny współczynnik rozgałęzienia z `b` do około `sqrt(b)`, co prowadzi do złożoności bliskiej `O(b^(d/2))`. W praktyce redukcja liczby odwiedzanych węzłów jest znacząca, co pozwala na przeszukiwanie drzewa na większą głębokość w tym samym czasie w porównaniu do czystego Minimax. Algorytm Alfa-Beta zawsze zwraca ten sam wynik co Minimax.

```py
def alphabeta(node, depth, alpha, beta, maximizingPlayer):
    if depth == 0 or node is a terminal node:
        return evaluate(node)

    if maximizingPlayer:
        value = -infinity
        for child in node.children():
            value = max(value, alphabeta(child, depth - 1, alpha, beta, False))
            alpha = max(alpha, value)
            if alpha >= beta:
                break
        return value
    else:
        value = +infinity
        for child in node.children():
            value = min(value, alphabeta(child, depth - 1, alpha, beta, True))
            beta = min(beta, value)
            if alpha >= beta:
                break
        return value
```

## Implementacja

### Kod gry Clobber

Kod napisany jest w języku Cython i generowany jest z niego osobny moduł

```py
from typing import Callable

from clobber.types cimport move

cdef class Board:

    def __init__(self, int n, int m, list state, int turn=0):
        self.n = n
        self.m = m
        self.state = list(state)
        self.turn = turn

    @staticmethod
    def initialize_board(int n, int m):
        state_py = []
        cdef int y
        cdef bint white_py

        for y in range(n):
            line_py = []
            white_py = (y % 2 == 0)
            for _ in range(m):
                line_py.append("W" if white_py else "B")
                white_py = not white_py
            state_py.append(" ".join(line_py))

        board = Board(n, m, state_py)
        return board

    def copy(self):
        return Board(self.n, self.m, [row for row in self.state], self.turn)

    def __str__(self):
        return "\n".join(self.state)

    def pretty(self):
        result_py = ["  " + " ".join(map(str, range(self.m)))]
        cdef int y
        for y in range(self.n):
            result_py.append(str(y) + " " + str(self.state[y]))
        return "\n".join(result_py)

    cpdef get_piece_at(self, tuple position):
        cdef int x, y
        x, y = position

        if x < 0 or y < 0 or y >= self.n or x >= self.m:
            return 'outside'

        row_str = <str>self.state[y]
        pieces = row_str.split(' ')
        return pieces[x]


    cpdef replace_piece_at(self, tuple position, object new_piece):
        cdef int x, y
        x, y = position

        if x < 0 or y < 0 or y >= self.n or x >= self.m:
            return 'outside'

        row_str = <str>self.state[y]
        pieces = row_str.split(' ')
        if x < len(pieces):
            pieces[x] = <str>new_piece
            self.state[y] = " ".join(pieces)
            return new_piece
        else:
            return 'error_index'

    def get_neighbours_positions(self, tuple position):
        return self.get_neighbours_positions_filtered(position, lambda p: p in ["B", "W"])

    cpdef list get_neighbours_positions_filtered(self, tuple position, object piece_filter):
        cdef int x, y
        cdef list neighbours_positions_py = []
        cdef tuple check_pos
        cdef object piece

        x, y = position

        positions_to_check = [(x - 1, y), (x + 1, y), (x, y - 1), (x, y + 1)]
        for check_pos in positions_to_check:
            piece = self.get_piece_at(check_pos)
            if piece_filter(piece):
                neighbours_positions_py.append(check_pos)

        return neighbours_positions_py

    def make_move(self, object new_color, move m):
        cdef tuple position_from, position_to
        position_from, position_to = m
        if self.get_piece_at(position_from) != new_color:
            raise Exception("make_move check failed: piece is not player's color")
        opponent_color = <object>("B" if new_color == 'W' else 'W')
        if self.get_piece_at(position_to) != opponent_color:
             raise Exception("make_move check failed: target is not opponent")

        self.replace_piece_at(position_from, '_')
        self.replace_piece_at(position_to, new_color)
        self.turn += 1

    def move_assuming_correct(self, object new_color, move m):
        cdef tuple position_from, position_to
        position_from, position_to = m
        self.replace_piece_at(position_from, '_')
        self.replace_piece_at(position_to, new_color)
        self.turn += 1

    cpdef list get_all_pieces(self, object piece):
        cdef list positions_py = []
        cdef int x, y
        for y in range(self.n):
            for x in range(self.m):
                if self.get_piece_at((x, y)) == piece:
                    positions_py.append((x, y))
        return positions_py

    def generate_moves(self, bint for_white):
        cdef dict possible_moves_py = {}
        cdef object opponent, current_piece
        cdef int x, y
        cdef list neighbouring_opponents_py

        opponent = <object>('B' if for_white else 'W')
        current_piece = <object>('W' if for_white else 'B')

        for y in range(self.n):
            for x in range(self.m):
                if self.get_piece_at((x, y)) != current_piece:
                    continue

                def opponent_filter(p):
                    return p == opponent

                neighbouring_opponents_py = self.get_neighbours_positions_filtered(
                    (x, y), opponent_filter)

                if not neighbouring_opponents_py:
                    continue

                possible_moves_py[(x, y)] = neighbouring_opponents_py

        return possible_moves_py

    cpdef bint has_moves(self, bint white):
        return len(self.generate_moves(white)) > 0
```

### Dodatkowe pliki Cython przydatne do poprawnego typowania itd.

```py
# board.pxd

cdef class Board:
    cdef public int n
    cdef public int m
    cdef public int turn
    cdef public list state

    cpdef get_piece_at(self, tuple position)
    cpdef replace_piece_at(self, tuple position, object new_piece)
    cpdef list get_neighbours_positions_filtered(self, tuple position, object piece_filter)
    cpdef list get_all_pieces(self, object piece)
    cpdef bint has_moves(self, bint white)
    
# board.pyi

from typing import Callable, List, Tuple, Dict, Any
from clobber.types import piece_type, move


class Board:
    n: int
    m: int
    turn: int
    state: List[str]

    def __init__(self, n: int, m: int,
                 state: List[str], turn: int = 0) -> None: ...

    @staticmethod
    def initialize_board(
        n: int, m: int) -> 'Board': ...

    def copy(self) -> 'Board': ...

    def __str__(self) -> str: ...
    def pretty(self) -> str: ...

    def get_piece_at(self, position: Tuple[int, int]) -> piece_type: ...

    def replace_piece_at(
        self, position: Tuple[int, int], new_piece: piece_type) -> piece_type: ...

    def get_neighbours_positions(
        self, position: Tuple[int, int]) -> List[Tuple[int, int]]: ...
    def get_neighbours_positions_filtered(self, position: Tuple[int, int], piece_filter: Callable[[
                                          piece_type], bool]) -> List[Tuple[int, int]]: ...

    def make_move(self, new_color: piece_type, m: move) -> None: ...
    def move_assuming_correct(
        self, new_color: piece_type, m: move) -> None: ...

    def get_all_pieces(self, piece: piece_type) -> List[Tuple[int, int]]: ...
    def generate_moves(
        self, for_white: bool) -> Dict[Tuple[int, int], List[Tuple[int, int]]]: ...

    def has_moves(self, white: bool) -> bool: ...

# types.pxd

ctypedef tuple move

# types.py

from typing import Literal

piece_type = Literal['B'] | Literal['W'] | Literal['_'] | Literal['outside']
move = tuple[tuple[int, int], tuple[int, int]]
```

Gra działa, mając reprezentację planszy jako lista stringów ("B" lub "W" lub "_"). Pozwala na kopiowanie, uzyskanie czytelnej postaci, sąsiadów danej pozycji (ze wsparciem funkcyjnym na filtr), poruszania się pionkami czy generację możliwych ruchów.

### Testy Clobber

In [1]:
import pytest

from clobber.board import Board


@pytest.mark.parametrize(("dimensions", "expected"),
                         [
    ((5, 6), "W B W B W B\nB W B W B W\nW B W B W B\nB W B W B W\nW B W B W B"),
    ((10, 10), "W B W B W B W B W B\nB W B W B W B W B W\nW B W B W B W B W B\nB W B W B W B W B W\nW B W B W B W B W B\nB W B W B W B W B W\nW B W B W B W B W B\nB W B W B W B W B W\nW B W B W B W B W B\nB W B W B W B W B W")
])
def test_board_generation(dimensions: tuple[int, int], expected: str) -> None:
    board: Board = Board.initialize_board(*dimensions)

    assert str(board) == expected


def test_neighbours() -> None:
    board: Board = Board.initialize_board(5, 6)

    black_neighbours: list[tuple[int, int]
                           ] = board.get_neighbours_positions_filtered((1, 1), lambda p: p == 'B')
    assert (0, 1) in black_neighbours
    assert (2, 1) in black_neighbours
    assert (1, 2) in black_neighbours
    assert (1, 0) in black_neighbours
    assert len(black_neighbours) == 4
    white_neighbours: list[tuple[int, int]
                           ] = board.get_neighbours_positions_filtered((1, 1), lambda p: p == 'W')
    assert len(white_neighbours) == 0


def test_get_piece() -> None:
    board: Board = Board.initialize_board(5, 6)

    assert board.get_piece_at((5, 4)) == 'B'
    assert board.get_piece_at((5, 5)) == 'outside'
    assert board.get_piece_at((6, 4)) == 'outside'


def test_generate_moves() -> None:
    board: Board = Board.initialize_board(2, 2)

    moves: dict[tuple[int, int], list[tuple[int, int]]
                ] = board.generate_moves(True)

    assert moves[(0, 0)] == [(1, 0), (0, 1)]
    assert moves[(1, 1)] == [(0, 1), (1, 0)]


def test_replace_piece() -> None:
    board: Board = Board.initialize_board(2, 2)

    board.replace_piece_at((0, 0), '_')
    assert board.get_piece_at((0, 0)) == '_'

    board.replace_piece_at((1, 1), '_')
    assert board.get_piece_at((1, 1)) == '_'

    board.replace_piece_at((0, 0), 'B')
    assert board.get_piece_at((0, 0)) == 'B'

    board.replace_piece_at((1, 0), 'B')
    assert board.get_piece_at((1, 0)) == 'B'


def test_move() -> None:
    board: Board = Board.initialize_board(5, 6)

    board.move_assuming_correct('W', ((0, 0), (1, 0)))

    assert board.get_piece_at((0, 0)) == '_'
    assert board.get_piece_at((1, 0)) == 'W'


def test_get_all_pieces() -> None:
    board: Board = Board.initialize_board(3, 3)

    white_positions: list[tuple[int, int]] = board.get_all_pieces('W')
    black_positions: list[tuple[int, int]] = board.get_all_pieces('B')

    assert len(white_positions) == 5
    assert len(black_positions) == 4
    assert (0, 0) in white_positions
    assert (1, 0) in black_positions


def test_has_moves() -> None:
    board: Board = Board.initialize_board(2, 2)

    assert board.has_moves(True)
    assert board.has_moves(False)

    board.replace_piece_at((0, 0), '_')
    board.replace_piece_at((1, 0), '_')
    board.replace_piece_at((0, 1), '_')
    board.replace_piece_at((1, 1), '_')

    assert not board.has_moves(True)
    assert not board.has_moves(False)


def test_copy() -> None:
    board: Board = Board.initialize_board(2, 2)
    board_copy: Board = board.copy()

    assert str(board) == str(board_copy)

    board_copy.replace_piece_at((0, 0), '_')
    assert board.get_piece_at((0, 0)) == 'W'
    assert board_copy.get_piece_at((0, 0)) == '_'


def test_turn() -> None:
    board: Board = Board.initialize_board(5, 5)

    assert board.turn == 0
    board.move_assuming_correct('W', ((0, 0), (0, 1)))
    assert board.turn == 1
    new_board: Board = board.copy()
    assert new_board.turn == 1
    board.move_assuming_correct('W', ((0, 1), (1, 0)))
    assert new_board.turn == 1
    assert board.turn == 2


test_neighbours()
test_get_piece()
test_generate_moves()
test_replace_piece()
test_move()
test_get_all_pieces()
test_has_moves()
test_copy()
test_turn()

### Implementacja Alfa-Beta <a name='Implementacja-Alfa-Beta'></a>

Opisz, jak zmodyfikowano funkcję Minimax, aby wprowadzić cięcia Alfa-Beta:
*   Dodatkowe parametry `alpha` i `beta`.
*   Inicjalizacja `alpha = -infinity`, `beta = +infinity` w pierwszym wywołaniu.
*   Przekazywanie `alpha` i `beta` w dół rekurencji.
*   Aktualizacja `alpha` w węzłach MAX i `beta` w węzłach MIN.
*   Implementacja warunków cięcia (`if alpha >= beta: break`).

### Funkcje pomocnicze dla heurystyk

W grze clobber nie jest trywialne określić kto wygrywa przez większość gry. W określeniu tego może pomóc podział gry na mniejsze "podgry", czyli obszary, gdzie połączone są ze sobą pionki obu drużyn.

In [2]:
from clobber.types import piece_type, move


def get_subgames(board: Board) -> list[list[tuple[tuple[int, int], piece_type]]]:
    """
    Identifies independent connected components (subgames/blocks) on the board.
    Returns a list of subgame representations (list of (Point, piece_type)).
    """
    rows: int = board.n
    cols: int = board.m
    visited = set()
    subgames: list[list[tuple[tuple[int, int], piece_type]]] = []

    for r in range(rows):
        for c in range(cols):
            start_point: tuple[int, int] = (r, c)
            piece: piece_type = board.get_piece_at(start_point)
            if piece != 0 and start_point not in visited:
                current_subgame_points: list[tuple[tuple[int, int], piece_type]] = [
                ]
                queue: list[tuple[int, int]] = [start_point]
                visited.add(start_point)
                has_black = False
                has_white = False

                while queue:
                    point: tuple[int, int] = queue.pop(0)
                    current_piece: piece_type = board.get_piece_at(point)
                    current_subgame_points.append((point, current_piece))
                    if current_piece == 'B':
                        has_black = True
                    if current_piece == 'W':
                        has_white = True

                    for neighbor in board.get_neighbours_positions(point):
                        if neighbor not in visited:
                            visited.add(neighbor)
                            queue.append(neighbor)

                if has_black and has_white:
                    subgames.append(current_subgame_points)

    return subgames


### Heurystyki

Konkretne heurystyki dziedziczą po abstrakcyjnej klasie Heurystyka, która dla danej planszy i dlakogo jest ruch wylicza wartość heurystyki.

- Random - jedyna niedeterministyczna heurystyka, używana tylko do początkowych testów - wybiera ruch losowo
- PieceSafetyHeuristic - oblicza, ile pionków gracza może zostać zbitych vs ile pionków przeciwnika
- CenterControlHeuristic - zakłada, że pionki w centrum mają większe znaczenie
- SubGameControlHeuristic - sprawdza, w ilu subgame'ach ma przewagę gracz, a w ilu przeciwnik
- LocalActivityHeuristic - pomijając ambicje przeciwnika, maksymalizuje liczbę możliwych zbić dla gracza

In [3]:
import random
from abc import ABC, abstractmethod

class Heuristic(ABC):
    @abstractmethod
    def calculate(self, board: Board, for_white: bool) -> float:
        ...


class Random(Heuristic):
    def calculate(self, board: Board, for_white: bool) -> float:
        return random.uniform(-10000, 10000)


class PieceSafetyHeuristic(Heuristic):
    '''
    Measures how many of your pieces can be captured by the opponent versus how many opponent pieces can be captured by you
    '''

    def calculate(self, board: Board, for_white: bool) -> float:
        my_player: piece_type = 'W' if for_white else 'B'
        opponent_player: piece_type = 'B' if for_white else 'W'

        my_pieces: list[tuple[int, int]] = board.get_all_pieces(my_player)
        opponent_pieces: list[tuple[int, int]
                              ] = board.get_all_pieces(opponent_player)

        my_vulnerable_count = 0
        for my_pos in my_pieces:
            my_vulnerable_count += 1 if board.get_neighbours_positions_filtered(
                my_pos, lambda p: p == opponent_player) else 0

        opponent_vulnerable_count = 0
        for opp_pos in opponent_pieces:
            opponent_vulnerable_count += 1 if board.get_neighbours_positions_filtered(
                opp_pos, lambda p: p == my_player) else 0

        score = float(opponent_vulnerable_count - my_vulnerable_count)
        return score * (1 if for_white else -1)


class CenterControlHeuristic(Heuristic):
    '''
    Heuristic assumes pieces should move towards center
    '''

    def calculate(self, board: Board, for_white: bool) -> float:
        my_player: piece_type = 'W' if for_white else 'B'
        opponent_player: piece_type = 'B' if for_white else 'W'

        center_x: float = (board.m - 1) / 2.0
        center_y: float = (board.n - 1) / 2.0

        my_score = 0.0
        for x, y in board.get_all_pieces(my_player):
            dist_sq: float = (x - center_x)**2 + (y - center_y)**2
            # Add small epsilon to avoid division by zero if piece is exactly center
            my_score += 1.0 / (1.0 + dist_sq + 1e-6)

        opponent_score = 0.0
        for x, y in board.get_all_pieces(opponent_player):
            dist_sq = (x - center_x)**2 + (y - center_y)**2
            opponent_score += 1.0 / (1.0 + dist_sq + 1e-6)

        score: float = my_score - opponent_score
        return score * (1 if for_white else -1)


class SubgameControlHeuristic(Heuristic):
    """
    Evaluates board state based on the piece difference (own - opponent)
    summed across all active subgames (subgames with both colors).
    """

    def calculate(self, board: Board, for_white: bool) -> float:
        my_player: piece_type = 'W' if for_white else 'B'
        opponent_player: piece_type = 'B' if for_white else 'W'

        active_subgames: list[list[tuple[tuple[int, int], piece_type]]] = get_subgames(board)
        total_score = 0.0

        if not active_subgames:
            return 0.0

        for sub_game in active_subgames:
            my_pieces = 0
            opponent_pieces = 0
            for _, piece in sub_game:
                if piece == my_player:
                    my_pieces += 1
                elif piece == opponent_player:
                    opponent_pieces += 1
            total_score += (my_pieces - opponent_pieces)

        return float(total_score) * (1 if for_white else -1)


class LocalActivityHeuristic(Heuristic):
    """
    Evaluates board state based on the number of opponent pieces adjacent
    to *our* pieces. More adjacent opponents means more potential future captures.
    """

    def calculate(self, board: Board, for_white: bool) -> float:
        my_player: piece_type = 'W' if for_white else 'B'
        opponent_player: piece_type = 'B' if for_white else 'W'

        total_activity_score = 0.0
        my_piece_positions: list[tuple[int, int]] = board.get_all_pieces(
            my_player)

        for my_pos in my_piece_positions:
            adjacent_opponents: int = len(board.get_neighbours_positions_filtered(
                my_pos, lambda p: p == opponent_player))
            total_activity_score += adjacent_opponents

        return float(total_activity_score) * (1 if for_white else -1)


### Agenci - MiniMax

Konkretni agenci dziedziczą po abstrakcyjnej klasie Agent. 

- Human: tym agentem rusza się gracz
- MiniMax: implementacja tego algorytmu
- AlphaBeta: MiniMax z cięciem alfa beta
- Dynamic: AlphaBeta z możliwością zmiany heurystyki na podstawie aktualnej tury

Dzięki wykorzystaniu takiej struktury danych trywialne było spełnienie rozszerzonej wersji zadania - wystarczy zdefiniować różnych agentów dla graczy w jednej potyczce.

In [4]:
from dataclasses import dataclass

import logging
import time


class Agent(ABC):
    def __init__(self, color: piece_type) -> None:
        self.color: piece_type = color
        self.opponent_color: piece_type = "B" if color == "W" else "W"
        self.total_time: float = 0
        self._total_nodes: float = 0

    @staticmethod
    def add_total_time(func):
        def wrapper(self, *args, **kwargs):
            start = time.time()
            result = func(self, *args, **kwargs)
            end = time.time()
            self.total_time += end - start
            return result
        return wrapper

    @abstractmethod
    @add_total_time
    def generate_move(self, board: Board) -> move:
        ...

    @property
    def total_nodes(self) -> float:
        return self._total_nodes


class Human(Agent):
    def generate_move(self, board: Board) -> move:
        print(board.pretty())
        print(f"your move, {self.color}")
        while True:
            try:
                x = int(input("piece start position x: "))
                y = int(input("piece start position y: "))

                if board.get_piece_at((x, y)) == 'outside':
                    print("outside board!")
                    continue

                if board.get_piece_at((x, y)) != self.color:
                    print("not your piece!")
                    continue

                x2 = int(input("piece end position x: "))
                y2 = int(input("piece end position y: "))

                if (x2, y2) not in board.get_neighbours_positions_filtered((x, y), lambda p: p == self.opponent_color):
                    print(
                        f"illegal move! {x2},{y2} not in {x},{y} {self.opponent_color} neighbours")
                    continue

                return ((x, y), (x2, y2))
            except Exception as e:
                print("invalid move! exception handled: ")
                print(e)


class MiniMax(Agent):
    def __init__(self, color: piece_type, heuristic: Heuristic, max_depth: int = 10) -> None:
        super().__init__(color)

        self.heuristic: Heuristic = heuristic
        self.max_depth: int = max_depth

    def minimax(self, board: Board, depth, is_maximizing) -> float:
        self._total_nodes += 1

        if depth >= self.max_depth:
            return self.heuristic.calculate(board, is_maximizing)

        if is_maximizing:
            best_score: float = float("-inf")
            for position_from, moves in board.generate_moves(True).items():
                for position_to in moves:
                    new_board: Board = board.copy()
                    new_board.move_assuming_correct(
                        'W', (position_from, position_to))

                    score: float = self.minimax(new_board, depth + 1, False)
                    best_score = max(score, best_score)
            return best_score

        best_score = float("inf")
        for position_from, moves in board.generate_moves(False).items():
            for position_to in moves:
                new_board = board.copy()
                new_board.move_assuming_correct(
                    'B', (position_from, position_to))

                score = self.minimax(new_board, depth + 1, True)
                best_score = min(score, best_score)

        return best_score

    @Agent.add_total_time
    def generate_move(self, board: Board) -> move:
        if self.color == 'W':
            best_score: float = float("-inf")
            best_move: move
            for position_from, moves in board.generate_moves(True).items():
                for position_to in moves:
                    new_board: Board = board.copy()
                    new_board.move_assuming_correct(
                        'W', (position_from, position_to))

                    score: float = self.minimax(new_board, 0, False)
                    if score >= best_score:
                        best_score = score
                        best_move = (
                            position_from, position_to)
            return best_move

        best_score = float("inf")
        for position_from, moves in board.generate_moves(False).items():
            for position_to in moves:
                new_board = board.copy()
                new_board.move_assuming_correct(
                    'B', (position_from, position_to))

                score = self.minimax(new_board, 0, True)
                if score <= best_score:
                    best_score = score
                    best_move = (
                        position_from, position_to)

        return best_move


class AlphaBeta(Agent):
    def __init__(self, color: piece_type, heuristic: Heuristic, max_depth: int = 10) -> None:
        super().__init__(color)

        self.heuristic: Heuristic = heuristic
        self.max_depth: int = max_depth

    def minimax(self, board: Board, depth, is_maximizing: bool, alpha: float, beta: float) -> float:
        self._total_nodes += 1

        if depth >= self.max_depth:
            return self.heuristic.calculate(board, is_maximizing)

        if is_maximizing:
            best_score: float = float("-inf")
            for position_from, moves in board.generate_moves(True).items():
                for position_to in moves:
                    new_board: Board = board.copy()
                    new_board.move_assuming_correct(
                        'W', (position_from, position_to))

                    score: float = self.minimax(
                        new_board, depth + 1, False, alpha, beta)
                    best_score = max(score, best_score)
                    alpha = max(alpha, best_score)

                    if beta <= alpha:
                        return best_score

            return best_score

        best_score = float("inf")
        for position_from, moves in board.generate_moves(False).items():
            for position_to in moves:
                new_board = board.copy()
                new_board.move_assuming_correct(
                    'B', (position_from, position_to))

                score = self.minimax(new_board, depth + 1, True, alpha, beta)
                best_score = min(score, best_score)
                beta = min(beta, best_score)

                if beta <= alpha:
                    return best_score

        return best_score

    @Agent.add_total_time
    def generate_move(self, board: Board) -> move:
        if self.color == 'W':
            best_score: float = float("-inf")
            best_move: move
            for position_from, moves in board.generate_moves(True).items():
                for position_to in moves:
                    new_board: Board = board.copy()
                    new_board.move_assuming_correct(
                        'W', (position_from, position_to))

                    score: float = self.minimax(
                        new_board, 0, False, float("-inf"), float("inf"))
                    if score >= best_score:
                        best_score = score
                        best_move = (
                            position_from, position_to)
            logging.debug(f"white heuristic: {best_score}")
            return best_move

        best_score = float("inf")
        for position_from, moves in board.generate_moves(False).items():
            for position_to in moves:
                new_board = board.copy()
                new_board.move_assuming_correct(
                    'B', (position_from, position_to))

                score = self.minimax(new_board, 0, True,
                                     float("-inf"), float("inf"))
                if score <= best_score:
                    best_score = score
                    best_move = (
                        position_from, position_to)

        logging.debug(f"black heuristic: {best_score}")
        return best_move


@dataclass(frozen=True)
class MinTurnRule():
    min_turn: int
    heuristic: Heuristic
    max_depth: int


class Dynamic(Agent):
    def __init__(self, color: piece_type, rules: list[MinTurnRule]) -> None:
        super().__init__(color)

        self.rules: list[MinTurnRule] = rules
        self.agent = AlphaBeta(color, rules[0].heuristic, rules[0].max_depth)

    @Agent.add_total_time
    def generate_move(self, board) -> move:
        current_turn: int = board.turn
        index = 0
        for rule in self.rules[1:]:
            if rule.min_turn > current_turn:
                break
            index += 1

        self.agent.heuristic = self.rules[index].heuristic
        self.agent.max_depth = self.rules[index].max_depth

        return self.agent.generate_move(board)

    @property
    def total_nodes(self) -> float:
        return self.agent.total_nodes


## 5. Wyniki i Ewaluacja <a name='Wyniki-i-Ewaluacja'></a>

### Przykładowa Rozgrywka <a name='Przykładowa-Rozgrywka'></a>

Pokaż wynik działania programu dla wersji podstawowej:
*   Plansza startowa (jeśli inna niż domyślna).
*   Wybrana heurystyka i głębokość `d`.
*   Końcowa reprezentacja planszy.
*   Informacja o liczbie rund i zwycięzcy.

*(Tutaj można wkleić wyjście programu)*

In [5]:
import sys


LOG_STEPS = False


def game(white: Agent, black: Agent, board: Board, print_logs: bool = False) -> piece_type:
    if print_logs:
        print("board initialized")
        print(board.pretty())

    current_color: piece_type = 'B'
    while True:
        current_player: Agent = white if current_color == 'W' else black
        chosen_move: move = current_player.generate_move(board)
        board.move_assuming_correct(current_color, chosen_move)
        if LOG_STEPS:
            print(
                f"{board.turn}. {current_color} moved from {chosen_move[0]} to {chosen_move[1]}")
            print(board.pretty())

        if not board.has_moves(current_color != 'W'):
            if print_logs:            
                print()
                print(board.pretty())
                print(f"{board.turn} turns taken")
                print(f"{current_color} won this game!")
                print(
                    f"white: {white.total_nodes} nodes visited, {white.total_time}s spent", file=sys.stderr,
                    flush=True)
                print(
                    f"black: {black.total_nodes} nodes visited, {black.total_time}s spent", file=sys.stderr,
                    flush=True)
                
            return current_color

        current_color = 'B' if current_color == 'W' else "W"


def main() -> None:
    rules: list[MinTurnRule] = [MinTurnRule(0, Random(), 1), MinTurnRule(
        10, SubgameControlHeuristic(), 10), MinTurnRule(20, LocalActivityHeuristic(), 10)]
    rules2: list[MinTurnRule] = [MinTurnRule(0, Random(), 1), MinTurnRule(
        3, SubgameControlHeuristic(), 5), MinTurnRule(10, LocalActivityHeuristic(), 15)]
    white: Agent = Dynamic('W', rules)
    black: Agent = Dynamic('B', rules2)
    board: Board = Board.initialize_board(5, 6)

    game(white, black, board, print_logs=True)


main()


board initialized
  0 1 2 3 4 5
0 W B W B W B
1 B W B W B W
2 W B W B W B
3 B W B W B W
4 W B W B W B


white: 594492 nodes visited, 11.288172245025635s spent
black: 6656534 nodes visited, 182.4768500328064s spent



  0 1 2 3 4 5
0 _ _ B _ _ B
1 _ W _ _ _ _
2 _ _ _ W _ _
3 _ _ _ _ B _
4 W _ _ _ _ B
23 turns taken
B won this game!


In [14]:
from tqdm import tqdm

def tournament(heuristics, games_per_pairing: int = 1, board_size: tuple[int, int] = (5, 6)) -> None:
    results = {heuristic.__class__.__name__: {"white": 0, "black": 0, "total_nodes": 0, "total_time": 0.0} for heuristic in heuristics}

    total_games: int = len(heuristics) ** 2 * games_per_pairing
    with tqdm(total=total_games, desc="Tournament Progress") as pbar:
        for white_heuristic in heuristics:
            for black_heuristic in heuristics:
                white_name = white_heuristic.__class__.__name__
                black_name = black_heuristic.__class__.__name__

                for _ in range(games_per_pairing):
                    white_agent = AlphaBeta('W', white_heuristic, max_depth=4)
                    black_agent = AlphaBeta('B', black_heuristic, max_depth=4)
                    board: Board = Board.initialize_board(*board_size)

                    winner = game(white_agent, black_agent, board)
                    if winner == 'W':
                        results[white_name]["white"] += 1
                    elif winner == 'B':
                        results[black_name]["black"] += 1

                    results[white_name]["total_nodes"] += white_agent.total_nodes
                    results[white_name]["total_time"] += white_agent.total_time
                    results[black_name]["total_nodes"] += black_agent.total_nodes
                    results[black_name]["total_time"] += black_agent.total_time

                    pbar.update(1)

    print("Tournament Results:")
    for heuristic_name, result in sorted(results.items(), key=lambda i: i[1]["white"] + i[1]["black"]):
        print(f"{heuristic_name}: {result['white']} wins as white, {result['black']} wins as black")
        print(f"  Total nodes visited: {result['total_nodes']}")
        print(f"  Total time spent: {result['total_time']:.2f} seconds")

heuristics = [
    PieceSafetyHeuristic(),
    CenterControlHeuristic(),
    SubgameControlHeuristic(),
    LocalActivityHeuristic()
]

tournament(heuristics)

Tournament Progress: 100%|██████████| 16/16 [21:40<00:00, 81.31s/it]

Tournament Results:
PieceSafetyHeuristic: 1 wins as white, 0 wins as black
  Total nodes visited: 7130389
  Total time spent: 218.54 seconds
SubgameControlHeuristic: 1 wins as white, 3 wins as black
  Total nodes visited: 8719551
  Total time spent: 399.64 seconds
LocalActivityHeuristic: 2 wins as white, 3 wins as black
  Total nodes visited: 11473665
  Total time spent: 210.02 seconds
CenterControlHeuristic: 2 wins as white, 4 wins as black
  Total nodes visited: 26611794
  Total time spent: 472.72 seconds


## 6. Opis Wersji Rozszerzonej (jeśli zaimplementowano) <a name='Opis-Wersji-Rozszerzonej'></a>

Jeśli zrealizowano wersję rozszerzoną (dodatkowe 20 pkt):
*   Opisz, jak umożliwiono grę między dwoma agentami (np. uruchomienie dwóch instancji programu, komunikacja, różne strategie/heurystyki dla każdego).
*   Jeśli zaimplementowano strategie adaptacyjne (zmiana heurystyki w trakcie gry), opisz ich działanie.

## 7. Napotkane Problemy Implementacyjne <a name='Napotkane-Problemy'></a>

Opisz krótko główne trudności napotkane podczas implementacji, np.:
*   Poprawna obsługa stanu gry (kopiowanie).
*   Debugowanie rekurencyjnych algorytmów Minimax/Alfa-Beta.
*   Projektowanie efektywnych i sensownych heurystyk.
*   Zarządzanie wydajnością dla większych głębokości.
*   Logika warunków końca gry.
*   Implementacja cięć Alfa-Beta (częste źródło błędów).

## 9. Materiały Źródłowe (Bibliografia) <a name='Bibliografia'></a>

Wymień wszystkie wykorzystane źródła:
*   Linki z treści zadania:
    *   [Clobber](https://link.do.opisu.clobber) (wstaw poprawny link, jeśli go masz)
    *   [Clobber - opis zasad](https://link.do.zasad.clobber) (wstaw poprawny link)
    *   J.F. Nordstrom – Introduction to Game Theory (lub inne pozycje z listy)
    *   M. Maschler et al., Game Theory
    *   N. Nisan et al., Algorithmic Game Theory
    *   M. J. Osborne, An Introduction to Game Theory
*   Notatki z wykładu.
*   Inne strony internetowe, artykuły, książki, tutoriale (jeśli były używane).

## Opis użycia bibliotek

### Standardowe biblioteki

- abc - definiowanie klas i metod abstrakcyjnych
- dataclasses - do definiowa klas danych, których strukturę można zamrozić i łatwo porównywać
- random - do generowania losowych liczb, przydatne było tylko w jednej, testowej heurystyce (jedyna niedeterministyczna)

### Zewnętrzne biblioteki

- geopy - do obliczania odległości między przystankami
- pandas - do szybszego przetworzenia pliku CSV
- Cython - generowanie kodu C z kodu podobnego do języka Python
- pytest - testy poprawności działania gry Clobber